# 完整版工作流——从快跑到全管控


> **阶段二核心 · 建议时长 ~40 min · 前置要求：01.01~01.03（或同等经验）**
>
零基础篇用 Flash 轻量版交付了 add_custom 和 sigmoid_custom（add_custom 实测 11 分钟）：一个 Agent，自走全流程。本节不再复述流程细节，而是回答三个更根本的问题：**什么时候值得为治理付出时间成本？完整版为什么这样设计？开发者在其中扮演什么角色？**

本节附一段真实运行（Ascend 910B3 + OpenCode + CANN 9.1，cannbot 主仓 ops-direct-invoke v3.0.0，全程 **~3.6 小时**）的精编证据。

路线：**为什么两套 → 三大设计思想 → 决策者的三类事 → 实录精编 → 选型总结**。全程约 40 分钟；课后实践另计。

## 为什么还需要完整版

同一台机器、同一类算子需求的两次实录：Flash 版 11 分钟交付，完整版 3.6 小时。

两次实录相差约 20 倍（算子与流程不同，仅作量级参考），但那次 3.6 小时的真实运行里，**算子开发节点的执行者自己发现并修好了一个缺陷**：冒烟测试（交付前的最小快速验证）输出全 NaN，执行者加载 `ascendc-precision-debug` Skill，定位到 Select 指令源序颠倒，修复后 34/34 全部通过，**没有任何人介入**。

多出的约 20 倍时间，换来的是**机制保证的独立验收与证据链**：每个节点有验收者独立核证据（不代跑测试），失败自动重试，全程可审计。本节讲清这套机制为什么这样设计、开发者在其中扮演什么角色。

## 1. 为什么需要两套工作流

一句话原理：**治理强度应该随风险缩放**。Flash 版像自测代码：快，但出了问题只有自己知道；完整版像带评审的正式交付：慢，但每一步有独立验收、有证据可查。为原型付治理成本是浪费，为正式交付省治理成本是冒险。

判断该切完整版的信号：

| 信号 | 为什么 |
|---|---|
| 要交付给客户 / 进生产仓 | 错误的代价不再只由开发者承担 |
| 有精度指标或审计要求 | 需要可追溯的验收证据链 |
| 算子复杂（条件语义 / Cast 链 / 规约） | 实现出错概率高，需要独立评审拦截 |
| 首次与 Agent 协作开发 | 边界感（哪些放权、哪些拍板）尚未建立 |

**v3.0.0 的关键变化**：完整版的所有机制都由 **harness**（cannbot 主仓内置的工作流编排程序，按 YAML 流程定义逐节点拉起执行者与验收者）自动执行。开发者不需要手动跑 CP 门禁或组装报告，失败由框架自动重试。

## 2. 完整版的三大设计思想

<img src="./images/02.01_complete_workflow.png" alt="完整版工作流 v3.0.0：三角色 × 七节点 × 节点级验收对偶" width="700px">

**思想一：节点级验收对偶**，把独立验收做成机制。零基础篇 01.01 节说过编写代码的角色不能自行宣布测试通过，角色示例中提到 architect / developer / reviewer；v3.0.0 把它落实为**每个节点的固定结构**：executor（执行者，由 architect/developer 担任）交付 → verifier（验收者，即 01.01 节示例中 reviewer 的机制化角色）独立核证据。 验收失败时，verifier 的失败原因（≤500 字符）由 harness **自动注入下一次执行**，无需人工传递，框架原生重试（默认 3 次）。

**思想二：边界静默**，启动前拍板、运行中零交互。 用户决策不再散落在过程中，而是集中在**启动前问卷**（≤5 题，答完后提示“需求已确认，后续可以静默开发了”）和**整轮退出后**（PM 读结果、规划下一轮：省略可选节点、复用已验收产物）。运行中零交互，期间甚至可以离开，回来验收产物。这是拍板语义、放权执行的更彻底形态。

**思想三：过程与产物分离，多轮可追溯。** 流程产物（workflow YAML、验收报告、测试执行记录、构建日志、证据清单）落 `.cannbot/<任务名>/workflow<N>/`，交付物（代码/测试/文档）落目标仓 `operators/<算子名>/`。每轮独立目录，前轮证据可复用（如代码修复节点确认"仅注释差异"时，AST 级验证后直接复用设备证据不重跑）。**过程可审计、产物可提交。**

## 3. 作为决策者驱动它：三类决策事项

完整版中开发者的角色从观察者变为决策者：

**① 写一份含规则顺序的需求。** 实录的需求是 xlog1py：`z = x * log1p(y)`，两条规则且**顺序固定**：x==0 → z=0 先应用；y 为 NaN → z=y 后应用并覆盖。要写到这个粒度，是因为 x=0 且 y=NaN 时，输出是 NaN 还是 0 取决于两条规则的应用顺序。**规则顺序是语义的一部分**，不写清楚 Agent 就得猜。零基础篇的五要素在这里升级为要素 + 语义优先级。

**② 在启动前问卷拍板。** 实录中的四项拍板：dtype 范围（仅 fp32）、输入约定（等形 + host 物化广播：把广播形状的输入在 CPU 侧展开成与 x 同 shape 的完整张量再传入 kernel，省了 kernel 复杂度，代价见 02.02 节）、测试规模、架构选型（SIMD，即单指令多数据，Vector 引擎路径；备选 SIMT 在 910B3/dav-2201 上不可用）。每个拍板都是一行决策。注意“等形输入”这个当时省事的决定，**02.02 节的性能数据会重新审视它的代价**。答完后收到提示“需求已确认，后续可以静默开发了”，此后即可离开。

**③ 读验收报告再放行。** 每个节点的验收报告是放行依据。被 verifier 打回时，**失败原因自动注入下一次执行、harness 自动重跑**，既不是跳过门禁，也不是弃用完整版换 Flash。整轮退出（某节点重试耗尽）也不是失败：PM 会规划下一轮，省略可选节点、复用已验收产物。

## 4. 实录精编：一次 3.6 小时的完整运行

> 完整运行共 6 个节点全部 pass（课程开发实录，全套台账未随仓发布）；以下精编只保留**改变认知的节点**。

**自动重试的威力（三个节点各重试 1 次，全部自愈）。** 0.0 知识搜集、0.1 黑盒测试设计、1 测试工程开发三个节点各验收失败 1 次，harness 自动重跑，第二次全部通过。没有人介入。旧版（v2 CP 门禁时代）这里需要人工确认；v3.0.0 把重试下沉到了框架层。

**执行者自愈（节点 2 · 算子开发）。** 冒烟测试（交付前的最小快速验证）输出全 NaN → executor 加载 `ascendc-precision-debug` Skill → 定位到 F 段 Select 指令源序颠倒 → 修复 → 34/34 全部通过。验证者独立重算 29 项源码哈希，确认两轮构建产物一致。**执行者不只是写代码，还会自行调试。**

**白盒与修复的证据复用（节点 3→4）。** 节点 3 白盒测试设计依源码生成 6 个用例接入框架，暴露 0 个算子缺陷；节点 4 代码修复确认“仅注释差异”，verifier 用 AST（抽象语法树）归一化 + token 级序列 + 位级 diff **三重独立验证**确认行为零变化，直接复用节点 3 的全量设备证据，不重跑。这就是设计思想三的落地：**证据可复用，重复劳动可避免。**

**最终验收：40/40。** 黑盒 34 + 白盒 6，全部通过（L0 基础 5/5、L1 常规 23/23、L2 边界 6/6、WB 白盒 6/6（黑盒 34 = L0+L1+L2）），精度 rtol=1e-5/atol=1e-07 全达标。

## 5. 两版对比与使用建议

| | Flash 版 | 完整版 v3.0.0 |
|---|---|---|
| 耗时（本次实录） | ~11 分钟 | ~3.6 小时（含 3 次自动重试） |
| 执行者 | 单 Agent | PM + architect/developer/verifier |
| 验收 | 内置（阶段内） | 节点级：每节点 verifier 独立核证据 |
| 失败处理 | Agent 自检 | harness 自动重试（默认 3 次） |
| 用户介入 | 发需求后观察 | 启动前问卷 + 整轮退出后决策 |
| 产物 | 工程目录 | 工程目录 + `.cannbot/` 全套台账 |
| 适用 | 原型/自测 | 正式交付/审计/复杂算子 |

**学完怎么用**：日常自己写原型 → Flash；要交付给客户或进仓 → 完整版。两版构成**风险适配**的治理工具箱，而非替代关系。

## 6. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | Flash 是快跑，完整版是全管控：风险适配而非替代 |
| **工具** | 完整版 v3.0.0（三角色七节点 + harness 自动重试） |
| **认知框架** | 三大设计思想（验收对偶 / 边界静默 / 过程与产物分离）+ 决策者三类事 |
| **下一步** | [02.02](./02.02_performance_optimization.ipynb)：跑对了，那跑得快吗 |

## 7. 课后练习

> 练习答案：[`answer/02.01_answer.txt`](./answer/02.01_answer.txt)

1. **（单选题）** v3.0.0 中，某节点验收失败后 harness 的处理是？
   - A. 暂停等用户手动修复
   - B. 自动重试该节点，失败原因注入下一次执行提示词
   - C. 跳过该节点继续下游
   - D. 整个工作流立即作废

2. **（单选题）** “需求已确认，后续可以静默开发了”出现的时机与含义？
   - A. 工作流结束后，交付完成
   - B. 启动前问卷答完后，此后运行中零交互
   - C. 每个节点通过后
   - D. 审批门开启后

3. **（简答）** 节点 4 代码修复节点确认“仅注释差异”后，为什么不重跑全量测试而是复用节点 3 的设备证据？这体现了哪条设计思想？

<details><summary>📖 准备：完整版环境安装（首次使用前）</summary>

```bash
git clone --recurse-submodules --shallow-submodules https://gitcode.com/cann/cannbot.git
cd cannbot/plugins-official/ops-direct-invoke
bash init.sh project opencode ~/ws-advanced
```
另需 Node.js ≥ 20.11 和 tmux（完整版 harness 依赖），详见[阶段三环境要求](../README.md)。
</details>

4. **（编程实践 · 项目线第 1 站）** 用完整版工作流重新开发你在 01.02 节课后实践已交付的 `sigmoid_custom`（`sigmoid(x) = 1/(1 + exp(-x))`，fp32，等形输入，禁止 `AscendC::Sigmoid` 高阶指令）。这是贯穿 02.01~02.03 项目实操主线的起点，02.02 将为它调优，02.03 将为它走提交进仓库的流程。

   > **为什么做同一道题**：01.02 节已用 Flash 版开发过该算子，现在换完整版重做，亲身体验两版的差异（启动前问卷 vs 发完就跑；逐节点验收报告 vs 单 Agent 自检；~3.6 小时 vs ~11 分钟，均为课程实录值，实际视算子与模型而定）。完成后对比两版的产物结构与测试证据，记录选型判断。

## 8. 要点回顾

- **节点级验收对偶**：每节点 executor 交付 + verifier 独立核证据（不代跑测试），失败 ≤500 字符回传 → harness 自动重试（默认 3 次）；
- **边界静默**：启动前问卷（≤5 题）→ 提示可以静默开发 → 运行中零交互 → 整轮退出后 PM 规划下一轮（省略可选节点、复用已验收产物）；
- **过程与产物分离**：`.cannbot/<任务名>/workflow<N>/` 多轮目录 + 证据复用（AST/token/位级三重验证确认无行为差异时，设备证据直接复用不重跑）；
- **选型**：治理强度随风险缩放，原型 Flash，正式交付完整版；
- 实录关键数字：3.6h / 6 节点（第 7 节点“文档准备”本次实录未触发：目标仓为全新工程，无已有算子文档） / 3 次重试全部自愈 / 40-40（黑盒 34 + 白盒 6）。

**下一节**：算子正确之后，接下来看性能：[02.02 · 性能调优](./02.02_performance_optimization.ipynb)。